<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Deep Learning & Generative AI</br>
    2.Lab GAN Explorer: CNNs as Discriminators and Bias Detection</br>
  </strong>
  <emph>
    Student Name: [Complete with your name]</br>
    Date: [Fill up with the submission date]</br>
  </emph>
</div>

## Background

This lab explores how CNNs function as "critics" in generative AI systems. You'll work with a Generative Adversarial Network (GAN), modify its CNN-based discriminator, and analyze the model for biases. This hands-on experience demonstrates the dual nature of CNNs: not just as classifiers, but as quality evaluators guiding generation.

---

## Description

This lab is structured around two parts.

### Part 1: Understanding the GAN Discriminator

You will:
- Load Fashion-MNIST dataset and set up data pipelines
- Review a provided generator architecture
- **Implement** a CNN discriminator from scratch
- **Complete** the adversarial training loop
- **Visualize** training dynamics (loss curves, generated samples, discriminator confidence)
- **Experiment** by modifying the discriminator architecture and observing impact

**Key Deliverables:**
- Working discriminator implementation
- Complete GAN training loop
- Visualizations of training progress
- Before/after comparison of architecture modification
- Answers to 2 reflection questions

### Part 2: Bias Detection and Critical Analysis

You will:
- Analyze Fashion-MNIST class distribution
- Train a simple classifier to label generated images
- **Identify** which clothing categories are over/under-represented in generated samples
- **Document** 2-3 specific bias examples with evidence (e.g., class imbalance, quality variance, mode collapse)
- **Propose and test** one mitigation strategy (e.g., weighted sampling, conditional generation, architecture changes)
- **Reflect** on ethical implications of deploying biased generative models

**Key Deliverables:**
- Class distribution analysis (real vs. generated)
- 2-3 documented bias examples with:
  - Clear description
  - Quantitative evidence
  - Potential source analysis
  - Impact assessment
- One tested mitigation strategy with results
- Ethical reflection (3-4 sentences)

---

## Instructions

1. **Work through sections sequentially** - Part 1 must be completed before Part 2
2. **Complete all code cells marked with `# TODO`** - These are required for full credit
3. **Answer all reflection questions** - Use the designated markdown cells
4. **Document your findings** - Use clear visualizations and written analysis

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from collections import defaultdict
from tqdm import tqdm

from rich.console import Console
from rich.table import Table

console = Console()

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
console.print(f"Using device: {device}")

## Part 1: Understanding the GAN Discriminator

### 1.1 Load Fashion-MNIST Dataset

Fashion-MNIST contains 70,000 grayscale images (28x28) across 10 clothing categories.

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

train_dataset = datasets.FashionMNIST(
    root="/content/drive/MyDrive/Laboratories/data",
    train=True,
    download=True,
    transform=transform,
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

console.print(f"Training samples: {len(train_dataset)}")
console.print(f"Number of batches: {len(train_loader)}")

### 1.2 Define the Generator Network

The generator takes random noise (latent vector) and transforms it into a 28x28 image.

**This is provided for you - review the architecture carefully!**

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim

        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 7, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        z = z.view(z.size(0), z.size(1), 1, 1)
        return self.model(z)


# Test generator
generator = Generator(latent_dim=100).to(device)
test_noise = torch.randn(4, 100).to(device)
test_output = generator(test_noise)
console.print(f"Generator output shape: {test_output.shape}")

### 1.3 Define the CNN Discriminator Network

Complete the discriminator implementation.

The discriminator is a CNN that learns to distinguish between real and generated images.

#### Key Requirements:
- Input: 28x28 grayscale image (1 channel)
- Use Conv2d layers with stride=2 for downsampling
- Use LeakyReLU activation (negative_slope=0.2)
- Output: Single probability value (Sigmoid activation)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, num_layers=3, base_channels=64):
        super(Discriminator, self).__init__()
        self.base_channels = base_channels

        layers = []
        in_channels = 1

        for i in range(num_layers):
            out_channels = base_channels * (2**i)
            layers.append(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=4,
                    stride=2,
                    padding=1,
                    bias=False,
                )
            )
            layers.append(nn.LeakyReLU(0.2))
            in_channels = out_channels

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        layers.append(
            nn.Conv2d(
                in_channels,
                out_channels=1,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False,
            )
        )
        layers.append(nn.Sigmoid())

        self.model = nn.Sequential(*layers)

    def forward(self, img):
        return self.model(img).view(-1)


# Test discriminator
discriminator = Discriminator(num_layers=3, base_channels=64).to(device)
test_input = torch.randn(4, 1, 28, 28).to(device)
test_output = discriminator(test_input)
console.print(f"Discriminator output shape: {test_output.shape}")  # Should be [4]
console.print(f"Sample outputs: {test_output}")

#### REFLECTION QUESTION 1:

Compare this discriminator to a typical CNN classifier (e.g., for MNIST digit recognition):
- What's different about the final layer?
- Why do we use Sigmoid activation at the end instead of Softmax?
- What does the output represent?

**Your Answer:**

The difference between the discriminator and a typical CNN in the final layer lies in what it seeks to classify. In the discriminator, the output is binary, i.e., it determines whether the image is real or fake, as it is designed to differentiate between images generated by the GAN and real images. In contrast, a typical CNN is used for multi-class classification, such as the classification of digits in MNIST (0-9), and its output represents the probability that the image belongs to one of those classes.

Therefore, in the discriminator we use a Sigmoid activation in the final layer, as this converts the output to a value between 0 and 1. This allows the network to determine whether the image is real (1) or fake (0). If we used Softmax, it would not be appropriate, since this activation is designed for multi-class problems, generating a probability distributed among multiple classes, which is not necessary in the case of the discriminator, which only has two options

### 1.4 Training Setup

In [ ]:
# Hyperparameters
latent_dim = 100
lr = 0.0002
beta1 = 0.5
num_epochs = 10

generator = Generator(latent_dim).to(device)
discriminator = Discriminator(num_layers=3, base_channels=64).to(device)

criterion = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

console.print("Training setup complete!")

### 1.5 GAN Training Loop

Complete the adversarial training loop.

**Adversarial Training Process:**
1. **Train Discriminator**: Learn to distinguish real from fake
2. **Train Generator**: Learn to fool the discriminator
3. Repeat until equilibrium

In [ ]:
def train_gan(generator, discriminator, num_epochs, latent_dim, lr, beta1):

    # Storage for tracking
    history = {
        "d_loss": [],
        "g_loss": [],
        "d_real_acc": [],
        "d_fake_acc": [],
        "generated_samples": [],
    }

    # Fixed noise for visualization
    fixed_noise = torch.randn(64, latent_dim).to(device)

    for epoch in range(num_epochs):
        epoch_d_loss = 0
        epoch_g_loss = 0
        epoch_d_real_acc = 0
        epoch_d_fake_acc = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for i, (real_imgs, _) in enumerate(pbar):
            batch_size = real_imgs.size(0)
            real_imgs = real_imgs.to(device)

            real_labels = torch.ones(batch_size).to(device)
            fake_labels = torch.zeros(batch_size).to(device)

            # ---------------------
            #  Train Discriminator
            # ---------------------
            optimizer_D.zero_grad()
            real_output = discriminator(real_imgs)
            d_loss_real = criterion(real_output, real_labels)

            z = torch.randn(batch_size, latent_dim).to(device)
            fake_imgs = generator(z)
            fake_output = discriminator(fake_imgs.detach())
            d_loss_fake = criterion(fake_output, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            optimizer_D.step()

            # -----------------
            #  Train Generator
            # -----------------
            optimizer_G.zero_grad()
            z = torch.randn(batch_size, latent_dim).to(device)
            fake_imgs = generator(z)
            fake_output = discriminator(fake_imgs)
            g_loss = criterion(fake_output, real_labels)
            g_loss.backward()
            optimizer_G.step()

            # Track metrics
            epoch_d_loss += d_loss.item()
            epoch_g_loss += g_loss.item()
            epoch_d_real_acc += (real_output > 0.5).float().mean().item()
            epoch_d_fake_acc += (fake_output < 0.5).float().mean().item()

        # Store epoch metrics
        num_batches = len(train_loader)
        history["d_loss"].append(epoch_d_loss / num_batches)
        history["g_loss"].append(epoch_g_loss / num_batches)
        history["d_real_acc"].append(epoch_d_real_acc / num_batches)
        history["d_fake_acc"].append(epoch_d_fake_acc / num_batches)

        # Generate samples for visualization
        with torch.no_grad():
            samples = generator(fixed_noise).cpu()
            history["generated_samples"].append(samples)

        console.print(
            f"Epoch {epoch+1}: D_loss={history['d_loss'][-1]:.4f}, G_loss={history['g_loss'][-1]:.4f}"
        )

    console.print("Training complete!")
    return history

In [ ]:
history = train_gan(
    generator,
    discriminator,
    num_epochs=num_epochs,
    latent_dim=latent_dim,
    lr=lr,
    beta1=beta1,
)

### 1.6 Visualize Training Progress

In [ ]:
def plot_gan_history(history):
    """Plots GAN training history"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history["d_loss"], label="Discriminator Loss", color="red")
    axes[0].plot(history["g_loss"], label="Generator Loss", color="blue")
    axes[0].set_title("Discriminator and Generator loss over epochs")
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(history["d_real_acc"], label="Accuracy on real images", color="red")
    axes[1].plot(history["d_fake_acc"], label="Accuracy on fake images", color="blue")
    axes[1].set_title("Discriminator accuracy on real vs. fake images")
    axes[1].set_xlabel("Epochs")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_gan_history(history)

### 1.7 Visualize Generated Samples at Different Training Stages

In [ ]:
def show_generated_images(samples, epoch, n_images=16):
    """Display a grid of generated images."""
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle(f"Generated Samples - Epoch {epoch}", fontsize=14)

    for idx, ax in enumerate(axes.flat):
        img = samples[idx].squeeze().numpy()
        img = (img + 1) / 2  # Denormalize
        ax.imshow(img, cmap="gray")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


checkpoints = [0, num_epochs // 2, num_epochs - 1]
for checkpoint in checkpoints:
    show_generated_images(history["generated_samples"][checkpoint], checkpoint + 1)

### 1.8 Discriminator Confidence Analysis

In [ ]:
discriminator.eval()
real_scores = []
fake_scores = []

with torch.no_grad():
    all_real_imgs = []
    while len(all_real_imgs) < 1000:
        real_imgs, _ = next(iter(train_loader))
        all_real_imgs.append(real_imgs)

    real_imgs = torch.cat(all_real_imgs)[:1000]
    real_imgs = real_imgs.to(device)

    real_outputs = discriminator(real_imgs)
    real_scores.extend(real_outputs.view(-1).cpu().numpy())

    z = torch.randn(1000, latent_dim).to(device)
    fake_imgs = generator(z)

    fake_outputs = discriminator(fake_imgs)
    fake_scores.extend(fake_outputs.view(-1).cpu().numpy())

plt.figure(figsize=(10, 5))
plt.hist(real_scores, bins=50, alpha=0.7, label="Real Images", color="blue")
plt.hist(fake_scores, bins=50, alpha=0.7, label="Fake Images", color="red")
plt.title("Discriminator Scores Distribution")
plt.xlabel("Discriminator Score")
plt.ylabel("Frequency")
plt.legend(loc="upper right")
plt.show()

console.print(
    f"Average discriminator score for REAL images: {np.mean(real_scores):.3f}"
)
console.print(
    f"Average discriminator score for FAKE images: {np.mean(fake_scores):.3f}"
)

### 1.9 EXPERIMENT: Modify the Discriminator Architecture

Modify the discriminator and observe the impact.

Try ONE of these modifications:
- Change `num_layers` (2 or 4 instead of 3)
- Change `base_channels` (32 or 128 instead of 64)
- Add dropout layers

Train for 5 epochs and compare results.

In [ ]:
console.print("EXPERIMENT: Modified Discriminator")
console.print("=" * 50)

discriminator_mod = Discriminator(num_layers=2, base_channels=64).to(device)
generator_mod = Generator(latent_dim).to(device)

history_mod = train_gan(
    generator_mod,
    discriminator_mod,
    num_epochs=5,
    latent_dim=latent_dim,
    lr=lr,
    beta1=beta1,
)

In [ ]:
plot_gan_history(history_mod)

#### EXPERIMENT Findings:

**Modification Tested:** I modified the discriminator by reducing the number of layers from 3 to 2.

**Observed Impact:**

After making these changes, I noticed that the generator loss dropped to 0, meaning the generator was able to fool the discriminator. This could mean that the generator is training well, but it might also suggest that the discriminator is too weak to tell the difference between real and fake images. Since I used fewer layers in the discriminator, it wasn't as effective in distinguishing real from fake images, which made it easier for the generator to trick it.

---

## Part 2: Bias Detection and Critical Analysis

### 2.1 Dataset Analysis: Class Distribution

In [ ]:
class_counts = defaultdict(int)

for _, label in train_dataset:
    class_counts[label] += 1

plt.figure(figsize=(12, 5))
plt.bar(class_names, class_counts.values(), color="blue")
plt.title("Class Distribution in Fashion-MNIST")
plt.xlabel("Class")
plt.ylabel("Number of Samples")

plt.xticks(range(10), class_names, rotation=45)
plt.show()

### 2.2 Generation Quality Analysis by Class

We'll train a simple classifier to label generated images, then analyze which classes are generated most/least.

In [ ]:
# Simple classifier (provided)
class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [ ]:
def train_classifier(classifier, num_epochs, lr):

    history = {"train_loss": [], "train_accuracy": []}

    optimizer_SC = optim.Adam(classifier.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_accuracy = 0
        num_batches = len(train_loader)

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for i, (inputs, labels) in enumerate(pbar):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer_SC.zero_grad()
            outputs = classifier(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_SC.step()

            _, predicted = torch.max(outputs, 1)
            correct = (predicted == labels).sum().item()
            accuracy = correct / inputs.size(0)

            epoch_loss += loss.item()
            epoch_accuracy += accuracy

            pbar.set_postfix(loss=loss.item(), accuracy=accuracy)

        history["train_loss"].append(epoch_loss / num_batches)
        history["train_accuracy"].append(epoch_accuracy / num_batches)

        console.print(
            f"Epoch {epoch+1}: Loss={history['train_loss'][-1]:.4f}, Accuracy={history['train_accuracy'][-1]:.4f}"
        )

    console.print("Classifier training complete!")

    return classifier, history

In [ ]:
console.print("Training classifier...")

classifier = SimpleClassifier().to(device)

history = train_classifier(classifier, num_epochs=3, lr=0.001)

In [ ]:
def evaluate_generated_images(
    generator, classifier, latent_dim, num_samples, batch_size
):
    classifier.eval()
    generator.eval()

    generated_class_counts = defaultdict(int)

    with torch.no_grad():
        z = torch.randn(num_samples, latent_dim).to(device)
        fake_imgs = generator(z)

        fake_imgs = fake_imgs.to(device)
        outputs = classifier(fake_imgs)
        _, predicted_classes = torch.max(outputs, 1)

        for class_id in predicted_classes:
            generated_class_counts[class_id.item()] += 1

    return generated_class_counts

In [ ]:
generated_class_counts = evaluate_generated_images(
    generator, classifier, latent_dim, num_samples=5000, batch_size=64
)

In [ ]:
def plot_class_distributions(class_counts, generated_class_counts):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(class_counts.keys(), class_counts.values(), color="blue", alpha=0.7)
    axes[0].set_title("Real Images Class Distribution")
    axes[0].set_xticks(range(len(class_names)))
    axes[0].set_xticklabels(class_names, rotation=45)
    axes[0].set_ylabel("Percentage")

    axes[1].bar(
        generated_class_counts.keys(),
        generated_class_counts.values(),
        color="orange",
        alpha=0.7,
    )
    axes[1].set_title("Generated Images Class Distribution")
    axes[1].set_xticks(range(len(class_names)))
    axes[1].set_xticklabels(class_names, rotation=45)
    axes[1].set_ylabel("Percentage")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_class_distributions(class_counts, generated_class_counts)

In [ ]:
def show_class_distribution_comparison(class_counts, generated_class_counts):
    real_percentages = [
        100 * class_counts[i] / sum(class_counts.values()) for i in range(10)
    ]
    generated_percentages = [
        100 * generated_class_counts[i] / sum(generated_class_counts.values())
        for i in range(10)
    ]

    table = Table(show_header=True, header_style="bold green")
    table.add_column("Class", style="dim", width=15)
    table.add_column("Real %", justify="right", width=10)
    table.add_column("Generated %", justify="right", width=15)
    table.add_column("Difference", justify="right", width=15)

    for i in range(10):
        real_pct = real_percentages[i]
        generated_pct = generated_percentages[i]
        diff = generated_pct - real_pct
        table.add_row(
            class_names[i], f"{real_pct:.2f}", f"{generated_pct:.2f}", f"{diff:.2f}"
        )

    console.print(table)

In [ ]:
show_class_distribution_comparison(class_counts, generated_class_counts)

### 2.3 Identify Failure Modes and Bias Examples

Find and visualize 2-3 specific examples of bias or quality issues.

In [ ]:
failure_examples = defaultdict(list)

num_samples = 5000
batch_size = 64

with torch.no_grad():
    total_generated = 0
    while total_generated < num_samples:
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_imgs = generator(z)
        disc_scores = discriminator(fake_imgs)
        class_preds = torch.argmax(classifier(fake_imgs), dim=1)

        fake_imgs = fake_imgs.cpu()
        disc_scores = disc_scores.cpu()
        class_preds = class_preds.cpu()

        for img, cls, score in zip(fake_imgs, class_preds, disc_scores):
            score_value = score.item()

            if score_value < 0.3:
                failure_examples[int(cls.item())].append((img, score_value))

        total_generated += batch_size


classes_with_failures = [cls for cls, ex in failure_examples.items() if len(ex) > 0]

if not classes_with_failures:
    console.print("No low-quality examples (score < 0.3) found.")
else:
    console.print(f"Found failure examples in {len(classes_with_failures)} classes.")

    for cls in classes_with_failures[:10]:
        examples = failure_examples[cls][:5]
        fig, axes = plt.subplots(1, len(examples), figsize=(12, 2))
        fig.suptitle(f"Low-quality samples for class {cls}", fontsize=14)

        if len(examples) == 1:
            axes = [axes]

        for ax, (img, score) in zip(axes, examples):
            ax.imshow(img.squeeze(), cmap="gray")
            ax.set_title(f"Score={score:.2f}")
            ax.axis("off")

        plt.tight_layout()
        plt.show()

### 2.4 Bias Documentation

Document 2-3 specific bias examples with evidence.

#### BIAS EXAMPLE 1:

**Description:**

When reviewing the generated images, I noticed that the Sandal class had several samples with very low quality, with scores below 0.3. Even though the model seems to generate a similar number of each clothing type, it clearly struggles more when creating Sandal with good definition. This shows a quality bias.

**Evidence:**

From the 5,000 generated images, the Sandal class had several examples with low scores, being one of the most affected. The overall class distribution looks balanced, but the difference is that its images appear more blurry or less realistic.

**Potential Source:**

This might happen because some classes look quite similar, and the model has trouble telling them apart. It could also be that the model wasn’t trained enough or didn’t have enough good examples for that class in the dataset.

**Impact:**

If this model were used in a real application, it could generate unclear or confusing images of Sandal. This would make the system look less reliable or less useful for that category.

#### BIAS EXAMPLE 2:

**Description:**

On the other hand, I noticed that the Coat class has the worst distribution when being classified and also shows some quality issues. I’m not sure what this kind of bias is called, but it clearly has two problems, the model generates very few coats, and the ones it does generate don’t look good.

**Evidence:**

According to the results, the Coat class represents only 4.80% of the generated images, which is much lower than the expected 10%, and some of those few images had low-quality scores below 0.3.

**Potential Source:**

This bias may come from the similar visual patterns between coat and other classes (such T-Shirt, Shirts, Pullover), making it hard for the model to learn their differences.

**Impact:**

If used in a real application, this could lead to the model failing to represent coats properly or even confusing them with other clothing types, which would make the results less accurate and less trustworthy.

#### BIAS EXAMPLE 3 (Optional):

**Description:**

**Evidence:**

**Potential Source:**

**Impact:**

### 2.5 Proposed Mitigation Strategy

Choose ONE mitigation approach and test it.

#### MITIGATION STRATEGY:

**Approach:**

I decided to improve the training of the classifier by increasing the number of epochs. The idea is to give the model more time to learn the visual differences between similar classes like Shirt, T-shirt, and Coat.

**Implementation:**

I increased the training epochs from 3 to 10, keeping the same architecture and learning rate. This allows the classifier to go through the dataset more times and better capture small visual details that help separate similar clothing items.

**Expected Impact:**

With more training, the classifier should make fewer mistakes in confusing similar classes.

In [ ]:
print("MITIGATION TEST")
print("=" * 60)

classifier_mod = SimpleClassifier().to(device)

history = train_classifier(classifier_mod, num_epochs=10, lr=0.001)

In [ ]:
generated_class_counts_mod = evaluate_generated_images(
    generator, classifier_mod, latent_dim, num_samples=5000, batch_size=64
)

plot_class_distributions(class_counts, generated_class_counts_mod)

In [ ]:
show_class_distribution_comparison(class_counts, generated_class_counts_mod)

#### MITIGATION RESULTS:

**Effectiveness:**

I noticed a clear difference in the class distribution predicted by the classifier based on the generated images. For example, pullover went from 17.16% to 7.90%, which is much closer to the real distribution. Similarly, coat increased from 4.80% to 12.20%, also aligning better with the real data.

**Trade-offs:**

As a positive result, the classes are now more balanced compared to when the model was trained with only 3 epochs. However, one downside is that the model is still not very efficient. This could be because both the generator and the discriminator architectures can still be improved. Based on the training values, I noticed that the losses are not yet similar, meaning the generator is still learning faster than the discriminator, so the discriminator cannot correctly determine whether the images are real or fake.

### 2.6 Ethical Reflection

#### ETHICAL REFLECTION (3-4 sentences):

**What are the ethical implications of deploying a generative model with the biases you identified?**

_(Consider: Who might be harmed? How could biases be amplified? What responsibilities do developers have?)_

**Your reflection:**

I imagine a company using this kind of technology either to manage clothing inventory or to track what people are wearing for market research or security purposes. Because there are biases in the training and data, the predictions made by the model might not be fully accurate, which could lead to wrong decisions, affecting both the company and the end users, like customers. While this is not as critical as a model used for medical diagnoses, developers still have a responsibility to carefully select data, tune parameters, and train the model properly. Any overlooked errors could cause problems, so attention to detail is very important.